<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/ASX_Quartely_Trades.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade yfinance
!pip install  --upgrade pandas_ta
!pip install ta pandas_ta

In [31]:
import yfinance as yf
print(yf.__version__)
import pandas as pd
import numpy as np
import time
import ta
import requests
from datetime import datetime, timedelta
from scipy.stats import linregress

print("Libraries Installed!")

0.2.66
Libraries Installed!


In [32]:
# List of ETFs to analyze
#df_o = pd.read_csv('stock_list.csv')
df_raw = pd.read_csv('etf_list.csv')
df_raw = df_raw[df_raw['Type'].isin(['ASX'])]

etfs = df_raw['Asset'].to_list()
print(etfs)

print(len(etfs))

['WGX.AX', 'VAU.AX', 'LYC.AX', 'ILU.AX', 'GMD.AX', 'DYL.AX', 'GGP.AX', 'NEM.AX', 'PLS.AX', 'CMM.AX', 'LTR.AX', 'NST.AX', 'EMR.AX', 'RRL.AX', 'EVN.AX', 'PDN.AX', 'PRU.AX', 'MIN.AX', 'SFR.AX', 'RMS.AX', 'AAI.AX', 'S32.AX', 'IGO.AX', 'FMG.AX', 'RIO.AX', 'DNL.AX', 'BHP.AX', 'CIA.AX', 'NIC.AX', 'ORI.AX']
30


In [33]:

# Filter ETFs or stocks for liquidity
def filter_by_liquidity(etf_df, ticker_col="Asset", min_dollar_vol=25e6, lookback_days=30):
    liquid_etfs = []

    for ticker in etf_df[ticker_col]:
        try:
            # Fetch daily historical data
            data = yf.download(ticker, period=f"{lookback_days*2}d", interval="1d", auto_adjust=True)

            if data.empty:
                continue

            # Calculate dollar volume (Close × Volume)
            data["dollar_volume"] = data["Close"] * data["Volume"]

            # Calculate rolling average over lookback_days
            avg_dollar_volume = data["dollar_volume"].rolling(window=lookback_days).mean().iloc[-1]

            # Check liquidity condition
            if avg_dollar_volume >= min_dollar_vol:
                liquid_etfs.append(ticker)

        except Exception as e:
            print(f"Error fetching {ticker}: {e}")

    # Return filtered DataFrame
    return etf_df[etf_df[ticker_col].isin(liquid_etfs)]

# Example usage
df = pd.DataFrame({"Assets": etfs})
liquid_df = filter_by_liquidity(df, ticker_col="Assets")
df_o = df_raw[df_raw['Asset'].isin(liquid_df['Assets'])]
etfs = df_o['Asset'].to_list()
print("")
print(etfs)
print(len(etfs))



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********


['WGX.AX', 'VAU.AX', 'LYC.AX', 'ILU.AX', 'GMD.AX', 'GGP.AX', 'NEM.AX', 'PLS.AX', 'CMM.AX', 'NST.AX', 'RRL.AX', 'EVN.AX', 'PDN.AX', 'PRU.AX', 'MIN.AX', 'SFR.AX', 'RMS.AX', 'S32.AX', 'FMG.AX', 'RIO.AX', 'BHP.AX', 'ORI.AX']
22


# Classify Sector Stages

In [34]:

def weinstein_stage(df, sma_window=30):
    """Determine Weinstein stage using 30-week SMA and its slope."""
     # Handle MultiIndex columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df["SMA"] = df["Close"].rolling(window=sma_window).mean()

    # Compute linear regression slope on last N SMA points
    if len(df.dropna()) < sma_window:
        return None  # not enough data

    slope, _, _, _, _ = linregress(range(sma_window), df["SMA"].tail(sma_window))

    latest_price = df["Close"].iloc[-1]
    latest_sma = df["SMA"].iloc[-1]

    # Determine stage
    if latest_price > latest_sma and slope > 0:
        stage = "Stage 2 (Advancing)"
    elif latest_price < latest_sma and slope < 0:
        stage = "Stage 4 (Declining)"
    elif latest_price < latest_sma and slope > 0:
        stage = "Stage 1 (Basing)"
    elif latest_price > latest_sma and slope < 0:
        stage = "Stage 3 (Topping)"
    else:
        stage = "Transition"

    return stage, slope, latest_price, latest_sma


In [35]:
# Classify stocks into stages
results = []
for etf in etfs:
    df = yf.download(etf, period="3y", interval="1wk", auto_adjust=True)
    stage_info = weinstein_stage(df)
    if stage_info:
        stage, slope, price, sma = stage_info
        results.append({
            "ETF": etf,
            "Stage": stage,
            "SMA_Slope": slope,
            "Latest_Price": price,
            "30W_SMA": sma
        })

stages_df = pd.DataFrame(results).sort_values(by="SMA_Slope", ascending=False)
stages_df


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,ETF,Stage,SMA_Slope,Latest_Price,30W_SMA
4,NEM.AX,Stage 2 (Advancing),0.947814,149.960007,98.179760
1,LYC.AX,Stage 2 (Advancing),0.128536,19.240000,11.253000
6,CMM.AX,Stage 2 (Advancing),0.106125,14.570000,10.186333
9,EVN.AX,Stage 2 (Advancing),0.104880,11.670000,8.436667
19,ORI.AX,Stage 2 (Advancing),0.092131,21.540001,19.524842
7,NST.AX,Stage 2 (Advancing),0.089346,26.049999,19.702756
8,RRL.AX,Stage 2 (Advancing),0.065946,6.470000,4.766174
3,GMD.AX,Stage 2 (Advancing),0.059715,6.810000,4.539333
13,SFR.AX,Stage 2 (Advancing),0.044626,15.700000,11.632667
11,PRU.AX,Stage 2 (Advancing),0.033598,5.170000,3.762928


In [38]:
advancing_stocks= stages_df[stages_df["Stage"] == "Stage 2 (Advancing)"]
advancing_stocks

,ETF,Stage,SMA_Slope,Latest_Price,30W_SMA
4,NEM.AX,Stage 2 (Advancing),0.947814,149.960007,98.179760
1,LYC.AX,Stage 2 (Advancing),0.128536,19.240000,11.253000
6,CMM.AX,Stage 2 (Advancing),0.106125,14.570000,10.186333
9,EVN.AX,Stage 2 (Advancing),0.104880,11.670000,8.436667
19,ORI.AX,Stage 2 (Advancing),0.092131,21.540001,19.524842
7,NST.AX,Stage 2 (Advancing),0.089346,26.049999,19.702756
8,RRL.AX,Stage 2 (Advancing),0.065946,6.470000,4.766174
3,GMD.AX,Stage 2 (Advancing),0.059715,6.810000,4.539333
13,SFR.AX,Stage 2 (Advancing),0.044626,15.700000,11.632667
11,PRU.AX,Stage 2 (Advancing),0.033598,5.170000,3.762928


In [30]:

def compute_mansfield_rs(asset_df, benchmark_df, window):
    """Compute Mansfield Relative Strength (MRS) vs a benchmark."""
    # Handle MultiIndex columns
    if isinstance(asset_df.columns, pd.MultiIndex):
        asset_df.columns = asset_df.columns.get_level_values(0)
    if isinstance(benchmark_df.columns, pd.MultiIndex):
        benchmark_df.columns = benchmark_df.columns.get_level_values(0)

    # Align and clean data
    data = pd.DataFrame({
        "Asset": asset_df["Close"],
        "Benchmark": benchmark_df["Close"]
    }).reindex(asset_df.index.union(benchmark_df.index)).ffill().dropna()

    if len(data) < window:
        return None  # not enough data

    data["RS"] = data["Asset"] / data["Benchmark"]
    data["RS_MA"] = data["RS"].rolling(window=window, min_periods=window).mean()
    data["MRS"] = ((data["RS"] / data["RS_MA"]) - 1) * 100

    return data["MRS"].iloc[-1]  # latest MRS value


In [44]:

# ETF universe
etfs = advancing_stocks['ETF'].to_list()

# Benchmark
benchmark = yf.download("STW.AX", period="6mo", interval="1wk", auto_adjust=True)

[*********************100%***********************]  1 of 1 completed


In [49]:
# ---------- STAGE 1: 3-MONTH MRS (12 weeks) ----------
mrs_3m = {}
for etf in etfs:
    data = yf.download(etf, period="6mo", interval="1wk", auto_adjust=True)
    mrs_value = compute_mansfield_rs(data, benchmark, window=12)
    if mrs_value is not None:
        mrs_3m[etf] = mrs_value

mrs_3m_df = pd.DataFrame(list(mrs_3m.items()), columns=["ETF", "MRS_3M"])
mrs_3m_df = mrs_3m_df[mrs_3m_df["MRS_3M"] > 0].sort_values(by="MRS_3M", ascending=False)
print("\n✅ 3-Month Positive MRS ETFs:")
mrs_3m_df.reset_index(drop=True, inplace=True)
mrs_3m_df


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


✅ 3-Month Positive MRS ETFs:


,ETF,MRS_3M
0,WGX.AX,44.217031
1,GMD.AX,31.845375
2,CMM.AX,25.244643
3,NST.AX,25.029754
4,NEM.AX,24.821827
5,EVN.AX,24.064454
6,RRL.AX,24.052676
7,LYC.AX,23.898381
8,PRU.AX,22.177137
9,RMS.AX,19.107476


In [48]:
# ---------- STAGE 2: 1-MONTH MRS (4 weeks) ----------
selected_etfs = mrs_3m_df["ETF"].tolist()
mrs_1m = {}
for etf in selected_etfs:
    data = yf.download(etf, period="3mo", interval="1wk",auto_adjust=True)
    mrs_value = compute_mansfield_rs(data, benchmark, window=4)
    if mrs_value is not None:
        mrs_1m[etf] = mrs_value

mrs_1m_df = pd.DataFrame(list(mrs_1m.items()), columns=["ETF", "MRS_1M"]).sort_values(by="MRS_1M", ascending=False)
print("\n🔥 1-Month MRS Ranking Among 3M Positive ETFs:")
mrs_1m_df.reset_index(drop=True, inplace=True)
mrs_1m_df

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


🔥 1-Month MRS Ranking Among 3M Positive ETFs:


,ETF,MRS_1M
0,NEM.AX,10.772996
1,GMD.AX,10.456751
2,WGX.AX,10.200965
3,CMM.AX,7.447821
4,RRL.AX,6.836323
5,NST.AX,6.174204
6,EVN.AX,5.462175
7,PRU.AX,4.640617
8,LYC.AX,3.053394
9,SFR.AX,2.992737
